# Maize Yield Transfer Learning — Training
USA → Indonesia / Vietnam / Thailand
Model: CropYieldLSTM v2 (hidden=512, 2 layers)
Fine-tuning: per-country epoch caps (THA: 20 total, IDN: 40 total, VNM: 70 total)

In [ ]:
# 1. Clone or update repo
import os

repo_dir = '/kaggle/working/thesis_maize'
if os.path.exists(repo_dir):
    print("Repo exists — pulling latest...")
    os.chdir(repo_dir)
    os.system('git pull')
else:
    os.system(f'git clone https://github.com/alisulas/thesis-maize-v2.git {repo_dir}')
    os.chdir(repo_dir)

os.system('pip install -r requirements.txt -q')
print("Done. CWD:", os.getcwd())

In [14]:
# 2. Cell 2 — Copy .npz ke lokasi yang benar:
import os, shutil


src = '/kaggle/input/datasets/alisulashidayat/maize-yield-modis-tensors'
dst = '/kaggle/working/thesis_maize/data/processed/modis'
os.makedirs(dst, exist_ok=True) #folder tujuan


for f in ['usa_modis.npz', 'idn_modis.npz', 'vnm_modis.npz', 'tha_modis.npz']:
  shutil.copy(f'{src}/{f}', f'{dst}/{f}')
  print(f'Copied {f}')

Copied usa_modis.npz
Copied idn_modis.npz
Copied vnm_modis.npz
Copied tha_modis.npz


In [15]:
# Cell 3 — Cek yield data ada di repo:

import os

# Daftar file yang dicek
files_to_check = [
    'data/processed/usa/yield_usa_2003_2023.parquet',
    'data/processed/indonesia/yield_indonesia_province_2020_2024.csv',
    'data/processed/vietnam/yield_vietnam_province_1995_2023.csv',
    'data/processed/thailand/thailand_province_yield_2021_2023.csv',
]

base_path = '/kaggle/working/thesis_maize'

for p in files_to_check:
    full_path = f'{base_path}/{p}'
    exists = os.path.exists(full_path)
    status = "OK" if exists else "MISSING"
    print(f"{status:7} {p}")

OK      data/processed/usa/yield_usa_2003_2023.parquet
OK      data/processed/indonesia/yield_indonesia_province_2020_2024.csv
OK      data/processed/vietnam/yield_vietnam_province_1995_2023.csv
OK      data/processed/thailand/thailand_province_yield_2021_2023.csv


In [16]:
# 3. Verify tensors
import numpy as np
for country in ['usa', 'idn', 'vnm', 'tha']:
    d = np.load(f'data/processed/modis/{country}_modis.npz')
    print(f'{country.upper()}: X={d["X"].shape}, y={d["y"].shape}, yield=[{d["y"].min():.2f}, {d["y"].max():.2f}]')

USA: X=(32296, 46, 10), y=(32296,), yield=[0.00, 16.96]
IDN: X=(162, 46, 10), y=(162,), yield=[0.00, 7.68]
VNM: X=(1315, 46, 10), y=(1315,), yield=[1.48, 9.02]
THA: X=(126, 46, 10), y=(126,), yield=[2.00, 5.57]


In [ ]:
# 4. Train USA baseline v2 (hidden=512, patience=30, 200 epochs)
import os
os.chdir('/kaggle/working/thesis_maize')
!python src/training/train.py --config experiments/configs/usa_lstm_v2.yaml

In [ ]:
import shutil, os

ckpt = 'experiments/checkpoints/usa_lstm_v2/best_model.pt'
print("Exists:", os.path.exists(ckpt))

if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / 1024 / 1024
    print(f"Size: {size:.1f} MB")
    os.makedirs('/kaggle/working/outputs', exist_ok=True)
    shutil.copy(ckpt, '/kaggle/working/outputs/usa_lstm_v2_best.pt')
    print("Copied to outputs/")

In [ ]:
# 5. Fine-tune all ASEAN countries from v2 checkpoint
# Per-country epoch caps (in finetune.py COUNTRY_EPOCH_OVERRIDES):
#   THA: frozen=10, full=10  (2 train years, no val set → aggressive cap)
#   IDN: frozen=20, full=20  (4 train years, no val set → moderate cap)
#   VNM: frozen=20, full=50  (default — has val set, early stopping works)
!python src/transfer/finetune.py --country all \
    --pretrained experiments/checkpoints/usa_lstm_v2/best_model.pt

In [ ]:
# 6. Show final results
import pandas as pd

results = pd.read_csv('experiments/logs/transfer_results.csv')
print(results.to_string(index=False))

In [6]:
import pandas as pd
log = pd.read_csv('./thesis_maize/experiments/logs/usa_baseline_lstm_train_log.csv')
print(log.tail(5)[['epoch','train_loss','val_loss','val_r2']].to_string(index=False))

 epoch  train_loss  val_loss   val_r2
   103    1.754717  3.598251 0.438931
   104    1.745352  3.200180 0.503587
   105    1.715271  3.513285 0.453533
   106    1.719700  3.115678 0.514183
   107    1.714173  3.168822 0.506346


In [10]:
import glob, os

# Cari semua file .pt
pt_files = glob.glob('/kaggle/working/**/*.pt', recursive=True)
print("PT files:", pt_files)

# Cek isi checkpoints folder lebih detail
for root, dirs, files in os.walk('/kaggle/working/thesis_maize/experiments/checkpoints'):
  print(f"DIR: {root}")
  for f in files:
      print(f"  {f}")

# Cek transfer_results.csv ukuran sebenarnya
path = '/kaggle/working/thesis_maize/experiments/logs/transfer_results.csv'
print(f"\ntransfer_results size: {os.path.getsize(path)} bytes")
print(open(path).read())

PT files: []
DIR: /kaggle/working/thesis_maize/experiments/checkpoints
  .gitkeep

transfer_results size: 380 bytes
country,transfer_r2,transfer_rmse,scratch_r2,scratch_rmse,r2_improvement
idn,0.0589576959609985,1.1348079442977903,-0.2971351146697998,1.3323256969451904,0.3560928106307983
vnm,-0.1854866743087768,1.4585583209991455,-0.0918787717819213,1.399789333343506,-0.0936079025268554
tha,-0.026449203491210938,0.4363648593425751,-0.03428041934967041,0.4380262792110443,0.007831215858459473



In [ ]:
# 7. Copy all outputs for download
import shutil, glob, os

os.makedirs('/kaggle/working/outputs', exist_ok=True)

# Checkpoints
for ckpt_dir in glob.glob('experiments/checkpoints/*'):
    shutil.copytree(ckpt_dir, f'/kaggle/working/outputs/{os.path.basename(ckpt_dir)}', dirs_exist_ok=True)

# Logs
for csv in glob.glob('experiments/logs/*.csv'):
    shutil.copy(csv, '/kaggle/working/outputs/')

print('Output files:')
for f in sorted(glob.glob('/kaggle/working/outputs/**/*', recursive=True)):
    if os.path.isfile(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f'  {f}  ({size:.1f} MB)')